In [2]:
import os
import cv2

# ========================= CONFIGURATION =========================

IMAGE_DIR = r"..\data_Set\\validation\\images" 
# =================================================================

# Global variables to track mouse state and coordinates
drawing = False
ix, iy = -1, -1
x1, y1, x2, y2 = -1, -1, -1, -1

def draw_bbox(event, x, y, flags, param):
    global ix, iy, x1, y1, x2, y2, drawing

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y
        x1, y1, x2, y2 = x, y, x, y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            x2, y2 = x, y

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        x2, y2 = x, y

def main():
    global ix, iy, x1, y1, x2, y2, drawing
    
    if not os.path.exists(IMAGE_DIR):
        print(f"Error: The directory '{IMAGE_DIR}' does not exist. Check your path.")
        return

    parent_dir = os.path.dirname(IMAGE_DIR)
    labels_dir = os.path.join(parent_dir, "labels")
    os.makedirs(labels_dir, exist_ok=True)
    print(f"Labels directory: {labels_dir}")

    valid_exts = (".png", ".jpg", ".jpeg", ".bmp")
    images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])

    if not images:
        print(f"No images found in '{IMAGE_DIR}'.")
        return

    print("\n" + "="*40)
    print(" DIRECT GUI KEYBOARD CONTROLS (Use inside Image Window):")
    print("  - [Drag Mouse] : Draw bounding box")
    print("  - [c]          : CLEAR/RESET the drawn box")
    print("  - [d]          : DELETE image from disk")
    print("  - [Space/Enter]: SAVE coordinates & go to next")
    print("  - [Esc]        : QUIT")
    print("="*40 + "\n")

    cv2.namedWindow("Annotation Window")
    cv2.setMouseCallback("Annotation Window", draw_bbox)

    idx = 0
    while idx < len(images):
        img_name = images[idx]
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        base_no_ext = os.path.splitext(img_name)[0]
        txt_path = os.path.join(labels_dir, f"{base_no_ext}.txt")

        # Skip if already annotated
        if os.path.exists(txt_path):
            idx += 1
            continue

        if not os.path.exists(img_path):
            idx += 1
            continue

        frame = cv2.imread(img_path)
        if frame is None:
            idx += 1
            continue

        # Reset box coordinates for the new image
        x1, y1, x2, y2 = -1, -1, -1, -1
        skip_image = False

        while True:
            # Create a clean display copy of the image on every loop
            temp_img = frame.copy()

            # If coordinates are active, draw the red bounding box
            if x1 != -1 and y1 != -1 and x2 != -1 and y2 != -1:
                cv2.rectangle(temp_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

            cv2.imshow("Annotation Window", temp_img)
            
            # Wait for a key press (non-blocking, responsive)
            key = cv2.waitKey(10) & 0xFF

            # --- KEY BINDINGS ---
            
            # 'c' key: CLEAR box coordinates instantly
            if key == ord('c'):
                x1, y1, x2, y2 = -1, -1, -1, -1
                print("Cleared box.")

            # 'd' key: DELETE current image
            elif key == ord('d'):
                try:
                    cv2.destroyAllWindows() # close window temporarily for prompt
                    confirm = input(f"Are you sure you want to delete {img_name}? (y/n): ").lower().strip()
                    if confirm == 'y':
                        os.remove(img_path)
                        if os.path.exists(txt_path):
                            os.remove(txt_path)
                        print(f"Deleted {img_name} from disk.")
                        idx += 1
                        skip_image = True
                        # Recreate window for next round
                        cv2.namedWindow("Annotation Window")
                        cv2.setMouseCallback("Annotation Window", draw_bbox)
                        break
                    else:
                        print("Delete canceled.")
                        # Recreate window
                        cv2.namedWindow("Annotation Window")
                        cv2.setMouseCallback("Annotation Window", draw_bbox)
                except Exception as e:
                    print(f"Error deleting file: {e}")

            # Space (32) or Enter (13): SAVE and proceed
            elif key == 32 or key == 13:
                if x1 != -1 and x2 != -1:
                    # Order coordinates properly in case of reverse dragging
                    xmin = min(x1, x2)
                    ymin = min(y1, y2)
                    xmax = max(x1, x2)
                    ymax = max(y1, y2)
                    
                    try:
                        with open(txt_path, 'w') as f:
                            f.write(f"{xmin} {ymin} {xmax} {ymax}\n")
                        print(f"Saved: labels/{os.path.basename(txt_path)}")
                        idx += 1
                        skip_image = True
                        break
                    except Exception as e:
                        print(f"Error saving file: {e}")
                else:
                    print("No box drawn! Draw a box before pressing Enter/Space.")

            # Esc key (27): QUIT program
            elif key == 27:
                print("Exiting...")
                cv2.destroyAllWindows()
                return

        if skip_image:
            continue

    cv2.destroyAllWindows()

    # ========================= COMPLETENESS CHECK =========================
    print("\n--- DATASET COMPLETENESS CHECK ---")
    remaining_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])
    unlabeled_count = 0
    
    for img in remaining_images:
        base = os.path.splitext(img)[0]
        corresponding_lbl = os.path.join(labels_dir, f"{base}.txt")
        if not os.path.exists(corresponding_lbl):
            unlabeled_count += 1
            print(f"MISSING LABEL: {img}")

    if unlabeled_count == 0:
        print("Success! Every single image currently in your dataset has been labeled.")
    else:
        print(f"Attention: There are still {unlabeled_count} image(s) left without a label file.")


if __name__ == "__main__":
    main()

Labels directory: ..\data_Set\\validation\labels

 DIRECT GUI KEYBOARD CONTROLS (Use inside Image Window):
  - [Drag Mouse] : Draw bounding box
  - [c]          : CLEAR/RESET the drawn box
  - [d]          : DELETE image from disk
  - [Space/Enter]: SAVE coordinates & go to next
  - [Esc]        : QUIT

Saved: labels/00149629982.txt
Saved: labels/00182743849.txt
Saved: labels/00216324277.txt
Saved: labels/00269846176.txt
Saved: labels/00318830007.txt
Cleared box.
Saved: labels/00400064853.txt
Saved: labels/00646871293.txt
Deleted 00713600328.jpg from disk.
Saved: labels/00988620001.txt
Saved: labels/01028518429.txt
Saved: labels/01128128913.txt
Saved: labels/01593351578.txt
Saved: labels/01618949505.txt
Cleared box.
Cleared box.
Cleared box.
Saved: labels/01794910008.txt
Saved: labels/01891151076.txt
Saved: labels/02155578111.txt
Saved: labels/02246053785.txt
Saved: labels/02305154375.txt
Saved: labels/02635059922.txt
Saved: labels/02741151662.txt
Saved: labels/03005451152.txt
Saved: l